# The parameter with a posterior that is only the prior

The sampler converged, R-hat is 1.00, and the posterior for the saturation constant is a
lognormal centred where you put it, three units wide. It looks like a result. It is the
prior, returned unchanged, because the doses that were run cannot separate that parameter
from the amplitude — and nothing in the fit output says so.

The remedy is not a bigger sample. There is nothing in the data to sharpen: the likelihood is
genuinely flat along a direction in parameter space, and the honest report is of the
*combination* that is not flat.

`design.structural` answers "how precisely will this design pin each parameter down", which
presumes each parameter is pinned down at all. For a nonlinear model that presumption is
often false, and false in a *diagnosable* way: some combination of the parameters is
estimable and the parameters separately are not.

A saturating response measured only at low dose is the canonical case. There
$\alpha x / (k + x) \approx (\alpha/k)\,x$, and no amount of data at low dose separates
$\alpha$ from $k$, because doubling both changes nothing at all.

That last sentence is the whole method. In **log**-parameter coordinates a scaling like
"double both" is a *constant* direction, so the search for it is linear algebra: build
$S_{ij} = \partial f_i / \partial \log \theta_j$, and read its null space (the
symmetries) and its row space (what the design can see).

In [ ]:
import numpy as np

from axiom.core import Add, Data, Div, Likelihood, ModelSpec, Mul, Param, Prior, dimensionless
from axiom.design import (
    Combination, EstimabilityReport, Observation, Prescription, ProfileReport, Scaling,
    SensitivityMatrix, estimable_combinations, observation_from_model, prescribe_measurements,
    profile_combination, profile_likelihood, sensitivity_matrix, simulated_identifiability,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import AQUA, BLUE, CRITICAL, ORANGE, annotate, caption, curve_band, lines, mark_x, mark_y, shade

enable();  # every axiom result renders itself from here on

NONE = dimensionless()
dose = Data(name="dose", dimension=NONE)
hill = Mul(factors=(
    Param(name="alpha", dimension=NONE),
    Div(numerator=dose, denominator=Add(terms=(Param(name="k", dimension=NONE), dose))),
))
theta = {"alpha": 4.0, "k": 10.0}

## An observation is an expression, its data, and its noise

That one object covers every case: a whole panel's worth of rows for a fitted model's mean,
a single extra dose you are considering, or a quantity you do not currently measure but
could. `observation_from_model` builds one from a `ModelSpec`.

In [ ]:
low = Observation("low dose", hill, {"dose": np.linspace(0.01, 0.3, 12)}, noise_sd=0.05)
wide = Observation("wide dose", hill, {"dose": np.linspace(0.5, 60.0, 12)}, noise_sd=0.05)

scaling: Scaling = "log"
sensitivity: SensitivityMatrix = sensitivity_matrix([low], theta, scaling=scaling)
print(sensitivity.parameters, "| rows:", sensitivity.rows, "| zero columns:", sensitivity.zero_columns)
print(np.round(sensitivity.matrix[:4], 4))
print("information:\n", np.round(sensitivity.information(), 3))

In [ ]:
from axiom.core import value

fine = np.linspace(0.0, 60.0, 300)
pairs = {"α = 4, k = 10": {"alpha": 4.0, "k": 10.0},
         "α = 40, k = 100": {"alpha": 40.0, "k": 100.0},
         "α = 0.4, k = 1": {"alpha": 0.4, "k": 1.0}}
fig = lines(
    fine,
    {label: np.ravel(value(hill, data={"dose": fine}, params=th)) for label, th in pairs.items()},
    colors=(BLUE, ORANGE, AQUA),
    title="Three completely different parameter values",
    subtitle="the same saturating response — α scaled by ten, k scaled by ten, α/k held fixed",
    x_title="dose", y_title="response",
)
shade(fig, 0.0, 0.3, text="where the data are", color=CRITICAL, alpha=0.10)
caption(fig, "Inside the shaded strip the three curves are indistinguishable at any noise "
             "level: low-dose data sees only the ratio α/k. The parameters separate where the "
             "curves separate, and nowhere else — which is a statement about the design, "
             "available before the study.")

In [ ]:
low_zoom = np.linspace(0.0, 0.3, 120)
fig = lines(
    low_zoom,
    {label: np.ravel(value(hill, data={"dose": low_zoom}, params=th)) for label, th in pairs.items()},
    colors=(BLUE, ORANGE, AQUA),
    title="…zoomed to the doses that were actually run",
    subtitle="the same three curves over the low-dose design's range",
    x_title="dose", y_title="response",
)
caption(fig, "Three lines, and you can see one. No sample size fixes this, because there is "
             "nothing in the data to fix — the likelihood is genuinely flat along that "
             "direction.")

## The tolerance is the question you are asking

A singular value below `tolerance` times the largest counts as zero. At `1e-8` you are
asking a **structural** question — is this direction flat to machine precision? At `1e-2`
you are asking a **practical** one — is it a hundred times flatter than the stiffest
direction, so that no realistic sample separates it?

The same design gives different, both-correct answers at the two settings.

In [ ]:
rows = []
for tolerance in (1e-8, 1e-2, 3e-2):
    report = estimable_combinations([low], theta, tolerance=tolerance)
    rows.append(
        [f"{tolerance:g}", report.rank, str([c.describe() for c in report.symmetries]),
         str([c.render() for c in report.estimable])]
    )
table(rows, headers=("tolerance", "rank", "symmetries", "estimable"))

In [ ]:
report: EstimabilityReport = estimable_combinations([low], theta, tolerance=3e-2)
print(report.summary())
print("singular values:", np.round(report.singular_values, 6))
print("condition number:", round(report.condition_number, 1))
print("null basis:", np.round(report.null_basis, 4))
symmetry: Combination = report.symmetries[0]
print("symmetry exponents:", symmetry.exponents, "| score (flatness):", round(symmetry.score, 5))
print("estimable:", report.estimable[0].render(), "| score (noise amplification):",
      round(report.estimable[0].score, 3))
print("identified:", report.identified)

### Local, persistent, and structural

A flat direction at one $\theta$ may be an accident of that value. Pass more points — prior
draws are the natural choice — and `persistent_deficiency` counts what stayed flat at all of
them. It rules out "an accident of these values"; it does **not** rule out "a property of
this design". Only a direction that survives varying the design too is a symmetry of the
model itself.

`a * b * x` is such a model: no design anywhere separates `a` from `b`.

In [ ]:
product = Mul(factors=(Param(name="a", dimension=NONE), Param(name="b", dimension=NONE), dose))
anywhere = Observation("any design", product, {"dose": np.linspace(0.1, 10.0, 20)}, 0.1)
structural = estimable_combinations(
    [anywhere], {"a": 2.0, "b": 3.0}, at=({"a": 0.5, "b": 8.0}, {"a": 5.0, "b": 0.2})
)
print("points tried:", structural.n_points)
print("deficiency:", structural.deficiency, "| persistent:", structural.persistent_deficiency)
print(structural.summary())

## What to measure next

A rank statement is not yet a decision. `prescribe_measurements` takes candidates — another
dose, a later period, an intermediate quantity — and greedily picks the smallest set that
restores full rank, reporting which symmetry each one breaks. Greedy is not optimal, so the
candidates it considered are reported too.

In [ ]:
plan: Prescription = prescribe_measurements([low], [wide], theta, tolerance=3e-2)
print("add:", plan.added, "| rank", plan.rank_before, "->", plan.rank_after, "| complete:", plan.complete)
print("each addition breaks:", plan.broken)
print("considered:", plan.considered, "| still flat:", plan.still_flat)

In [ ]:
# When nothing on the table can help, that is the answer — and it is the useful one.
more_of_the_same = Observation("more of the same", product, {"dose": np.linspace(20.0, 30.0, 5)}, 0.1)
hopeless = prescribe_measurements([anywhere], [more_of_the_same], {"a": 2.0, "b": 3.0})
print("complete:", hopeless.complete, "| added:", hopeless.added, "| still flat:", hopeless.still_flat)

## Practical identifiability: run the experiment before you run it

The rank test is a statement about a derivative. `simulated_identifiability` runs the
proposed experiment — simulates the outcome at a known truth, refits, and profiles the
likelihood of each target — so "not identified" becomes an interval that is infinite on one
side rather than an eigenvalue near zero (Raue et al. 2009).

This catches what the rank test misses: a parameter identified in principle and unbounded in
practice at this noise level and this sample size.

In [ ]:
model = ModelSpec(
    name="hill",
    mean=hill,
    outcome=Data(name="y", dimension=NONE),
    likelihood=Likelihood(family="normal", scale="sigma"),
    parameters=(
        Param(name="alpha", dimension=NONE, prior=Prior(family="lognormal", hyper={"mu": 1.0, "sigma": 1.0})),
        Param(name="k", dimension=NONE, prior=Prior(family="lognormal", hyper={"mu": 2.0, "sigma": 1.0})),
        Param(name="sigma", dimension=NONE, prior=Prior(family="halfnormal", hyper={"sigma": 1.0})),
    ),
)
truth = {"alpha": 4.0, "k": 10.0, "sigma": 0.02}
print(observation_from_model(model, {"dose": np.array([1.0])}, theta=truth).noise_sd)

In [ ]:
ratio = Combination(exponents={"alpha": 1, "k": -1}, kind="estimable", score=0.0)
for label, doses in (("low dose", np.linspace(0.01, 0.3, 24)), ("wide dose", np.linspace(0.5, 60.0, 24))):
    profile: ProfileReport = simulated_identifiability(
        model, {"dose": doses}, truth,
        targets=["alpha", "k"], combinations=[ratio],
        seed=3, range_factor=30.0, n_grid=11,
    )
    print(f"--- {label} --- identified: {profile.identified}, flat: {profile.flat}")
    table(
        [
            [target, f"{profile.truth[target]:.4g}", f"{profile.recovered[target]:.4g}",
             f"[{profile.interval[target][0]:.4g}, {profile.interval[target][1]:.4g}]"]
            for target in profile.targets
        ],
        headers=("target", "truth", "mle", "95%"),
    )

Read the low-dose row again: **$\alpha$ and $k$ are each unbounded above, and $\alpha/k$ is
pinned to a few percent.** That is the finding, and it is the number the report should
quote — not a wide interval on $\alpha$, and certainly not a point estimate of it.
Two warnings about the printed profile of `alpha` in the next cell. The first is the finding:
its drops do not trace a profile, because on an exactly flat ridge each constrained fit is
warm-started from the previous one and the optimizer terminates without moving `k` — the
number it reports is an artifact of where it started. The second is the consequence: read the
interval from `simulated_identifiability` above, which refits from its own start and reports
`[-inf, inf]`, rather than from these drops.

The ratio's profile, by contrast, is a profile.

In [ ]:
# The profiles underneath, one target at a time.
rng = np.random.default_rng(3)
doses = np.linspace(0.01, 0.3, 24)
from axiom.core import value
simulated = {"dose": doses}
simulated["y"] = np.asarray(value(hill, data=simulated, params=truth)) + rng.normal(0, 0.02, 24)

grid = [1.0, 2.0, 4.0, 8.0, 16.0]
values, drops = profile_likelihood(model, simulated, truth, "alpha", grid=grid)
print("alpha profile:   ", [f"{v:>5g}->{d:6.3f}" for v, d in zip(values, drops)])

ratio_grid = [0.30, 0.35, 0.40, 0.45, 0.50]
values, drops = profile_combination(model, simulated, truth, ratio, grid=ratio_grid)
print("alpha/k profile: ", [f"{v:>5g}->{d:6.3f}" for v, d in zip(values, drops)])
print("(a drop of 1.92 is the 95% boundary; the ratio crosses it on both sides)")

In [ ]:
ratio_vals, ratio_drops = profile_combination(model, simulated, truth, ratio, grid=[0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6])
fig = curve_band(
    ratio_vals, ratio_drops,
    label="α / k",
    color=ORANGE,
    title="…and the combination that does",
    subtitle="the same data, profiling the ratio instead",
    x_title="α / k", y_title="drop in log likelihood",
)
mark_y(fig, 1.92, text="95% boundary")
mark_x(fig, truth["alpha"] / truth["k"], text="truth")
caption(fig, "Sharp, centred on the truth, and crossing the 95% boundary on both sides — "
             "which is what a profile of an estimable quantity looks like. This is what the "
             "study measured, and the number the report should carry: not a wide interval on "
             "α, and certainly not a point estimate of it.")

## When it will not answer

Log coordinates need positive parameters, and a monomial of a non-positive parameter is not
a quantity. `scaling="absolute"` drops that requirement and gives linear combinations
instead of monomials. Anything the analysis cannot do comes back as a typed `Unsupported`
naming what is missing, never as a number to be trusted.

In [ ]:
print(sensitivity_matrix([anywhere], {"a": 0.0, "b": 1.0}).reason)
print(simulated_identifiability(model, {"dose": doses}, {"alpha": 4.0, "k": 10.0}).missing)

## Which *observable* has to be measured

The two halves meet here. A latent compartment fills from the dose, and the outcome reads
it, scaled:

$$\texttt{state}[t] = \texttt{decay}\cdot\texttt{state}[t-1] + \texttt{uptake}\cdot\texttt{dose}[t],
\qquad \texttt{outcome}[t] = \texttt{gain}\cdot\texttt{state}[t]$$

Unroll it and `gain` and `uptake` only ever appear multiplied together. No dose schedule and
no number of periods separates them — but one measurement of the compartment does. Neither
module knows about the other; the compiled expression is the whole interface.

In [ ]:
from axiom.core import D
from axiom.dynamics import Variable, parse_system, time_ref, unroll

system = parse_system(
    """
    state   = decay * state[t-1] + uptake * dose
    outcome = gain * state
    """,
    variables=(
        Variable(name="state", dimension=D.currency, observed=False),
        Variable(name="outcome", dimension=D.outcome),
        Variable(name="dose", dimension=D.currency, role="exogenous"),
    ),
    parameters=(
        Param(name="decay", dimension=NONE),
        Param(name="uptake", dimension=NONE),
        Param(name="gain", dimension=D.outcome / D.currency),
    ),
    name="two-compartment",
)
compiled = unroll(system, periods=8)
rng = np.random.default_rng(4)
schedule = {time_ref("dose", t): rng.gamma(2.0, 1.0, size=6) for t in range(8)}
truth = {"decay": 0.6, "uptake": 2.0, "gain": 3.0}


def trajectory(variable, label):
    return [
        Observation(f"{label}@{t}", compiled.expression(variable, t), schedule, 0.1)
        for t in range(1, 8)
    ]

In [ ]:
outcome_only = estimable_combinations(
    trajectory("outcome", "outcome"),
    truth,
    at=({"decay": 0.3, "uptake": 1.0, "gain": 1.0}, {"decay": 0.8, "uptake": 4.0, "gain": 0.5}),
)
print("rank", outcome_only.rank, "of", len(outcome_only.parameters),
      "| persistent deficiency:", outcome_only.persistent_deficiency)
print("flat: ", [c.describe() for c in outcome_only.symmetries])
print("estimable:", [c.render() for c in outcome_only.estimable])

In [ ]:
# More of the same measurement cannot break a symmetry ...
more_outcome = prescribe_measurements(
    trajectory("outcome", "outcome")[:3], trajectory("outcome", "later outcome")[3:], truth
)
print("more outcome periods -> complete:", more_outcome.complete,
      "| still flat:", more_outcome.still_flat)

# ... one measurement of the latent compartment does.
measure_state = prescribe_measurements(
    trajectory("outcome", "outcome"), trajectory("state", "state")[:1], truth
)
print("measure the compartment -> add", measure_state.added,
      "| rank", measure_state.rank_before, "->", measure_state.rank_after,
      "| breaks", measure_state.broken)

That is the answer to "what else do I have to measure": not a bigger sample, not a longer
study — one reading of the quantity in the middle. The rank test says so before any data is
collected, and it says so in the model's own vocabulary rather than as an eigenvalue.

## Seeing it

`enable()` at the top of this notebook already made a bare result on the last
line of a cell render itself — a card drawn by `rich`, or the same content as
aligned plain text where `rich` is not installed. `show` does it on demand, for
a result that is not the last thing in its cell.

`axiom.viz` draws the figure this subpackage's results are actually about.

In [ ]:
from axiom.design import obrien_fleming
from axiom.display import show
from axiom.viz import boundary

rule = obrien_fleming(0.025, [0.25, 0.5, 0.75, 1.0], kind="efficacy")
show(rule)
boundary(rule)

Drawn as a step, because the threshold holds *until* the next look rather than sliding between them.

## What this bought you

The difference between "this parameter is uncertain" and "this parameter is not in the data",
before the study runs — plus the combination that *is* in the data, named in the model's own
vocabulary, and the smallest set of extra measurements that would break the symmetry. When
nothing on the table would, that is the answer too.